In [ ]:
# Cell 1: Installations
!pip install -q -U transformers peft accelerate bitsandbytes

import torch
import transformers

# Mute non-critical warnings
transformers.logging.set_verbosity_error() 
print("✅ Libraries installed and ready.")

In [ ]:
# Cell 2: Base Model Loading
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

BASE_MODEL_ID = "NousResearch/Meta-Llama-3-8B-Instruct"

print("Loading Base Model in 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

print("✅ Base Model and Tokenizer loaded into GPU.")

In [ ]:
# Cell 3: Attach the LoRA Adapter
from peft import PeftModel

ADAPTER_PATH = "/kaggle/input/models/armanhalavi/academic-humanizer-lora/pytorch/default/1/thesis_instruct_adapter_final"

print("Merging your custom LoRA Adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

print("✅ Pipeline fully assembled. Ready for inference.")

In [ ]:
# Cell 4: Gradio Web UI (Fixed)
!pip install -q gradio

import gradio as gr
import torch
import warnings

# THE FIX: Forcefully mute the Gradio 6.0 deprecation warning
warnings.filterwarnings("ignore", category=DeprecationWarning)

def rewrite_text(ai_draft):
    if not ai_draft.strip():
        return "Please enter some text."
        
    messages = [
        {
            "role": "system", 
            # THE FIX: Explicitly commanding burstiness and varied sentence structure
            #"content": "You are a professional researcher in statistics and data science. Rewrite the following text into dry, human-sounding academic prose. You MUST use high burstiness: alternate between very short, punchy sentences and longer, complex sentences. Do not use flowery transitions. DO NOT generate titles, headers, or introductory phrases. Output ONLY the rewritten paragraph."
            "content": """You are a professional researcher in statistics and data science. Rewrite the following text into dry, human-sounding academic prose. You MUST use high burstiness: alternate between very short, punchy sentences and longer, complex sentences. Use slightly unconventional but academically accurate phrasing. Do not use flowery or robotic transitions like 'furthermore' or 'in conclusion'."
            
            ### TARGET STYLE ANCHORS ###
            Mimic the exact sentence length variance, tone, and punctuation style of these human-written examples:
            
            Example 1:
            The cryptocurrency market has developed significantly since the launch of Bitcoin in January 2009, thereby providing new investment opportunities to agents. One important issue is the extent to which it can be characterized as an efficient market, which requires prices to follow a random walk and thus to be unpredictable. Whilst numerous studies have analyzed persistence of other asset prices (see, e.g., Mills, 1993; Barkoulas & Baum, 1996; Jacobsen, 1996; Caporale and Gil-Alana, 2004; Caporale et al., 2016) , the evidence concerning the cryptocurrency market is more limited. The most extensive study on this topic is due to Caporale et al. (2018) , who found timevarying persistence in Bitcoin, Litecoin, Wave, and Run over the period 2013-2017, whilst Caporale and Plastun (2019) detected an anomaly in the case of Bitcoin, which appears to have higher returns on Mondays than on the other days of the week.

            Example 2:
            Despite the relatively recent launch of cryptocurrencies, their considerable success has led to various studies being carried out on the properties of this new market. For instance, Giudici et al.(2020) highlighted some of the benefits of cryptocurrencies but also stressed the need for more stringent regulations. Papadimitriou et al. (2020) found that the cryptocurrencies with the highest market capitalization are not necessarily those offering the most profitable investment the MF-DFA method in addition to the Hurst exponent one and found long memory in six cryptocurrencies. Aslan and Sensoy (2020) again used the Hurst exponent for the most capitalized cryptocurrencies and found considerable heterogeneity across these markets. Finally, Arouxet et al. (2022) reported that the Covid-19 pandemic had only a slight impact on the long-memory properties of both the returns and volatility of seven cryptocurrencies.
            """
        },
        {
            "role": "user", 
            "content": f"Rewrite this:\n\n{ai_draft}"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=800, 
            do_sample=True,          
            temperature=0.75,        # Previous value = 0.85
            top_p=0.90,              # THE FIX: Nucleus sampling for vocabulary variety
            repetition_penalty=1.12, # Previous value = 1.18
            eos_token_id=terminators,
            pad_token_id=tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[1]
    response_ids = outputs[0][input_length:]
    human_rewrite = tokenizer.decode(response_ids, skip_special_tokens=True).strip()
    
    # THE FIX: A hardcoded Python cleanup for the tokenization spacing bugs
    human_rewrite = human_rewrite.replace(" )", ")").replace(" ,", ",").replace(" .", ".").replace(" ;", ";")

    return human_rewrite

# THE FIX: Theme is back in Blocks where Gradio 5.x demands it
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Academic Style-Transfer Interface")
    gr.Markdown("Paste your AI-generated drafts below. Your custom LoRA adapter will rewrite them into dry, professional statistics and data science prose.")
    
    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(
                lines=12, 
                label="Original AI Draft", 
                value="The transformative potential of analyzing historical price data alongside fundamental coin metadata cannot be overstated, as it unveils profound, paradigm-shifting insights into the underlying market dynamics.",
                placeholder="Paste the flowery AI text here..."
            )
            submit_btn = gr.Button("Humanize Text", variant="primary")
        
        with gr.Column():
            output_text = gr.Textbox(
                lines=12, 
                label="LoRA Academic Rewrite",
                interactive=False
            )
            
    submit_btn.click(fn=rewrite_text, inputs=input_text, outputs=output_text)

# THE FIX: launch() is clean again
demo.launch(share=True, debug=True)